# Heatmap scale experiment: common max/min ratios

Compare small **HTL** and **large** chambers using their current maxima, with the same logarithmic dynamic range in each candidate. The existing manuscript scales span factors of **3,000** (HTL) and **300** (large).

| Candidate | Small HTL min → max | Large min → max | Shared max/min |
| --- | --- | --- | --- |
| Current reference | 1e-6 → 3e-3 | 1e-6 → 3e-4 | unequal: 3,000 / 300 |
| 100 | 3e-5 → 3e-3 | 3e-6 → 3e-4 | 100 |
| 300 | 1e-5 → 3e-3 | 1e-6 → 3e-4 | 300 |
| 1,000 | 3e-6 → 3e-3 | 3e-7 → 3e-4 | 1,000 |

Holding maxima fixed preserves the current saturation threshold. A factor of 100 devotes the color range to two decades of probability; 1,000 spans three decades and reveals lower-density structure. Compare spatial detail, background speckling, and saturation before choosing. With these shifted ranges, equal colors indicate equal probability **relative to the chamber-specific maximum**, not equal absolute probability.

Run this notebook in the project's analysis environment, with the Synology videos mounted. It reads recipe definitions from `manuscript_figure_panels.ipynb` without running that notebook. It does not edit manuscript settings or copy experiment figures to manuscript destinations.


In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'analyze.py').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'analyze.py').is_file(), 'Start from the repository or notebooks directory'
OUT = ROOT / 'imgs' / 'heatmap_scale_experiment'
OUT.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.plotting.heatmap_cache import render_heatmap_cache
CACHE = OUT / 'cache'
CACHE.mkdir(exist_ok=True)

# Load only assignments defining these recipes; skip every run/copy statement.
recipe_namespace = {'shlex': shlex}
manuscript = json.loads((ROOT / 'notebooks/manuscript_figure_panels.ipynb').read_text())
for cell in manuscript['cells']:
    if cell['cell_type'] != 'code':
        continue
    assignments = []
    for node in ast.parse(''.join(cell['source'])).body:
        if isinstance(node, ast.Assign) and all(
            isinstance(target, ast.Name) and target.id.startswith((
                'FIGURE_4B_', 'figure_4b_',
                'EXTENDED_DATA_FIGURE_13A_', 'extended_data_figure_13a_',
            )) for target in node.targets
        ):
            assignments.append(node)
    if assignments:
        exec(compile(ast.Module(body=assignments, type_ignores=[]),
                     'manuscript heatmap recipe definitions', 'exec'), recipe_namespace)

RECIPES = {
    'small_flat_training': ('small', recipe_namespace['extended_data_figure_13a_flat_training']),
    'small_agarose_training': ('small', recipe_namespace['extended_data_figure_13a_agarose_training']),
    'small_flat_pre': ('small', recipe_namespace['extended_data_figure_13a_flat_pre']),
    'small_agarose_pre': ('small', recipe_namespace['extended_data_figure_13a_agarose_pre']),
    'large_training': ('large', recipe_namespace['figure_4b_training']),
    'large_post': ('large', recipe_namespace['figure_4b_post']),
}


## Select comparisons

Start with flat HTL Training 2 SB5 and large-chamber complete Training 2, matching the existing recipes. Their time windows and cohorts differ, so this is a comparison of color-scale appearance, not a controlled biological comparison between chamber sizes. Add pre-period, agarose, or post-period recipes from `RECIPES` to check whether a choice works across the manuscript.

The first default sweep runs **two full analyses**, one per selected recipe, and renders eight figures from cached probability maps. Subsequent scale changes only rerender. `RUN = False` previews commands; set it to `True` to build missing caches and render. `REBUILD_CACHE = True` forces fresh analyses; reset it afterward.

Caches are stored under `imgs/heatmap_scale_experiment/cache/`. Recipe options, local configuration, and analysis source code are recorded and checked before reuse. **After changing tracking data, input files, or externally maintained exclusions, set `REBUILD_CACHE = True`.** Cache reuse does not inspect the source recordings, so cached results can be rendered without the Synology mount. Existing PNGs alone do not count as a probability cache.

Each fresh analysis runs in a temporary directory to keep its ordinary statistics and plots away from manuscript outputs. All selected scale images are rerendered each time `RUN = True`, so new bounds and style settings take effect without reanalysis. The original manuscript notebook is never edited.


In [ ]:
SELECTED = ['small_flat_training', 'large_training']
RATIOS = [100, 300, 1000]
INCLUDE_REFERENCE = True
RUN = False
REBUILD_CACHE = False
PYTHON = sys.executable  # Must have analyze.py dependencies installed.
MAXIMA = {'small': 3e-3, 'large': 3e-4}
FONT_FAMILY = 'Arial'
FONT_SIZE = 20

assert len(SELECTED) == len(set(SELECTED)), 'Select each recipe only once'
assert SELECTED and all(name in RECIPES for name in SELECTED)
assert RATIOS and len(RATIOS) == len(set(RATIOS))
assert all(isinstance(ratio, (int, float)) and 1 < ratio < float('inf') for ratio in RATIOS)
assert all(0 < value < float('inf') for value in MAXIMA.values())
CANDIDATES = ([('reference', None)] if INCLUDE_REFERENCE else []) + [
    (f'ratio_{ratio:g}', ratio) for ratio in RATIOS
]


def scale_bounds(name, ratio):
    chamber, _ = RECIPES[name]
    maximum = MAXIMA[chamber]
    minimum = 1e-6 if ratio is None else maximum / ratio
    if not 0 < minimum < maximum:
        raise ValueError('Heatmaps require 0 < min < max')
    return minimum, maximum


def analysis_command(name, export_path):
    _, recipe = RECIPES[name]
    argv = shlex.split(recipe['command'])
    argv[0] = PYTHON
    argv[1] = str(ROOT / 'analyze.py')
    # Display limits are irrelevant to the exported, unclipped probability maps.
    for flag in ['--pltHmVmin', '--pltHmVmax']:
        index = argv.index(flag)
        del argv[index:index + 2]
    return argv + ['--imgFormat', 'png', '--fontFamily', FONT_FAMILY,
                   '--fs', str(FONT_SIZE), '--no-wall-cache',
                   '--hm-map-export', str(export_path)]


def cache_provenance(name):
    # Scale and style choices do not invalidate numeric map caches.
    argv = analysis_command(name, '<cache>')
    for flag in ['--fontFamily', '--fs', '--imgFormat']:
        index = argv.index(flag)
        del argv[index:index + 2]
    config = ROOT / '.analyze.local.env'
    digest = hashlib.sha256()
    sources = [ROOT / 'analyze.py', *sorted((ROOT / 'src/analysis').rglob('*.py')),
               *sorted((ROOT / 'src/utils').rglob('*.py'))]
    for source in sources:
        digest.update(str(source.relative_to(ROOT)).encode())
        digest.update(source.read_bytes())
    return {'version': 1, 'command': argv,
            'local_config': config.read_text() if config.exists() else None,
            'analysis_source_sha256': digest.hexdigest()}


for name in SELECTED:
    print(f'One analysis if cache is missing/stale: {name}')
    print(shlex.join(analysis_command(name, CACHE / f'{name}.npz')))
    for label, ratio in CANDIDATES:
        minimum, maximum = scale_bounds(name, ratio)
        display(Markdown(f'**{label} / {name}:** {minimum:g} → {maximum:g} '
                         f'(max/min = {maximum / minimum:g}); render from cache'))


In [ ]:
def ensure_cache(name):
    target = CACHE / f'{name}.npz'
    manifest = target.with_suffix('.json')
    provenance = cache_provenance(name)
    if target.is_file() and manifest.is_file() and not REBUILD_CACHE:
        try:
            matches = json.loads(manifest.read_text()) == provenance
        except (ValueError, OSError):
            matches = False
        if matches:
            print(f'Reusing probability maps: {target.name}')
            return target
    log = OUT / f'{name}_analysis.log'
    env = dict(os.environ)
    env['PYTHONPATH'] = str(ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    env['MPLBACKEND'] = 'Agg'
    with tempfile.TemporaryDirectory(prefix='analysis_', dir=OUT) as work:
        work = Path(work)
        (work / 'imgs').mkdir()
        config = ROOT / '.analyze.local.env'
        if config.exists():
            shutil.copy2(config, work / config.name)
        fresh_cache = work / 'probability_maps.npz'
        argv = analysis_command(name, fresh_cache)
        print(f'Analyzing {name}; progress log: {log}', flush=True)
        with log.open('w') as stream:
            stream.write(shlex.join(argv) + '\n')
            stream.flush()
            subprocess.run(argv, cwd=work, env=env, stdout=stream,
                           stderr=subprocess.STDOUT, check=True)
        if not fresh_cache.is_file():
            raise FileNotFoundError(f'No fresh probability maps generated; inspect {log}')
        # Replace the cache only after the full analysis succeeds.
        os.replace(fresh_cache, target)
    manifest.write_text(json.dumps(provenance, indent=2) + '\n')
    return target


if RUN:
    import matplotlib.pyplot as plt
    for name in SELECTED:
        cache_path = ensure_cache(name)
        for label, ratio in CANDIDATES:
            minimum, maximum = scale_bounds(name, ratio)
            target = OUT / f'{name}_{label}.png'
            fig = render_heatmap_cache(cache_path, minimum, maximum,
                                       font_family=FONT_FAMILY, font_size=FONT_SIZE)
            try:
                fig.savefig(target, bbox_inches='tight')
                fig.savefig(target.with_suffix('.pdf'), bbox_inches='tight')
            finally:
                plt.close(fig)
            target.with_suffix('.json').write_text(json.dumps({
                'recipe': name, 'candidate': label, 'vmin': minimum, 'vmax': maximum,
                'ratio': maximum / minimum, 'cache': str(cache_path),
                'cache_provenance': json.loads(cache_path.with_suffix('.json').read_text()),
            }, indent=2) + '\n')
            print(f'Rendered {target.name}')
else:
    print('Preview only. Set RUN = True to build missing caches and render the sweep.')


## Compare rendered heatmaps

Rows show candidates; columns show selected recipes. The reference row intentionally uses unequal ratios. Each image retains its own probability colorbar. Check whether low-density regions remain informative and whether the reward-centered peak is excessively saturated. Pick a ratio only after checking the additional periods/cohorts of interest.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

missing = [OUT / f'{name}_{label}.png'
           for label, _ in CANDIDATES for name in SELECTED
           if not (OUT / f'{name}_{label}.png').is_file()]
if not RUN:
    print('Preview only. Set RUN = True and run the rendering cell before comparing.')
elif missing:
    print(f'{len(missing)} figures still needed. Run the rendering cell first.')
else:
    fig, axes = plt.subplots(len(CANDIDATES), len(SELECTED), squeeze=False,
                             figsize=(6 * len(SELECTED), 7 * len(CANDIDATES)))
    for row, (label, ratio) in enumerate(CANDIDATES):
        for col, name in enumerate(SELECTED):
            ax = axes[row, col]
            ax.imshow(mpimg.imread(OUT / f'{name}_{label}.png'))
            minimum, maximum = scale_bounds(name, ratio)
            ax.set_title(f'{name} | {label}\n{minimum:g} to {maximum:g}')
            ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUT / 'comparison.png', dpi=160, bbox_inches='tight')
    fig.savefig(OUT / 'comparison.pdf', bbox_inches='tight')
    plt.show()
